# ガウス正規座標：断面の曲がり具合と、測地線の間隔

[ノート](../../../研究ノート/02_微分幾何/christoffel_riemann_intro.md#C-3-1)（付録C-3-1）の、外的曲率 $K_{ij}=-\tfrac12\partial_\tau g_{ij}$ を、動かして確かめます。

平坦な時空 $(y,t)$（計量 $dy^2-dt^2$）で、曲がった断面 $\Sigma:\ t=\kappa y^2$ を取り、断面の各点から**法線方向に測地線**（直線）を伸ばします。隣り合う測地線の間隔が、固有時間 $\tau$ とともにどう変わるかが、外的曲率 $K$ です。

**使い方**：下部のスライダーで、断面の曲がり具合 $\kappa$ を変えます。

In [ ]:
import numpy as np
import sympy as sp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 検算：$K=-\tfrac12\partial_\tau g$ を、SymPy で確かめる

断面 $X(y)=(y,\kappa y^2)$（成分は $(y,t)$）、計量 $G=\mathrm{diag}(1,-1)$、単位法線 $n\propto(2\kappa y,\ 1)$ から、測地線 $F(y,\tau)=X(y)+\tau\,n(y)$ を作ります。

- 外的曲率の定義：$K=G(X'',\,n)$
- ガウス正規座標の誘導計量：$g(y,\tau)=G(\partial_yF,\partial_yF)$

が、$K=-\tfrac12\partial_\tau g|_{\tau=0}$ を満たすことを確かめます。

In [ ]:
y, tau, kap = sp.symbols("y tau kappa", real=True)
G = sp.diag(1, -1)
ip = lambda p, q: sp.simplify((p.T * G * q)[0])
X = sp.Matrix([y, kap * y**2])
e = X.diff(y)
n = sp.Matrix([2 * kap * y, 1]) / sp.sqrt(1 - 4 * kap**2 * y**2)
assert sp.simplify(ip(n, e)) == 0 and sp.simplify(ip(n, n) + 1) == 0     # 法線は断面と直交し、ε = −1

K = ip(X.diff(y, 2), n)
F = X + tau * n
g_tau = sp.simplify(ip(F.diff(y), F.diff(y)))
dg = sp.simplify(g_tau.diff(tau).subs(tau, 0))
print("K =", sp.simplify(K))
print("g(y, τ) =", g_tau)
assert sp.simplify(K + dg / 2) == 0
print("K = −(1/2) ∂τ g が一致しました")
g0 = sp.simplify(g_tau.subs(y, 0))
print("y = 0 では g(τ) =", sp.factor(g0))
assert sp.simplify(g0 - (1 + 2 * kap * tau) ** 2) == 0

## スライダーで動かす

- **左**：$(y,t)$ の時空図です。黒が断面 $\Sigma$、青が法線方向の測地線、灰色の曲線が $\tau=0.5,1.0,\dots$ の断面です。
- **右**：$y=0$ の隣り合う測地線の間隔 $g(\tau)=(1+2\kappa\tau)^2$ です。点線は $\tau=0$ での接線で、傾きが $-2K=4\kappa$ です。
- $\kappa<0$ では、測地線がやがて交わります（焦点 $\tau=1/(2|\kappa|)$）。この点で $g=0$ となり、ガウス正規座標は、その手前までしか使えません。

In [ ]:
kappas = np.round(np.arange(-0.5, 0.501, 0.05), 2) + 0.0     # −0.00 の表示を避ける
ys = np.linspace(-0.8, 0.8, 9)
T = 1.6


def Fxy(kappa, y0, tau):
    nrm = np.sqrt(1 - 4 * kappa**2 * y0**2)
    return y0 + tau * 2 * kappa * y0 / nrm, kappa * y0**2 + tau / nrm


def gaps(rows):
    xs, zs = [], []
    for xx, zz in rows:
        xs += list(xx) + [None]; zs += list(zz) + [None]
    return xs, zs


def build(kappa):
    yy = np.linspace(-0.8, 0.8, 200)
    sigma = (yy, kappa * yy**2)
    taus = np.linspace(0, T, 50)
    geo = gaps([Fxy(kappa, y0, taus) for y0 in ys])
    slices = gaps([Fxy(kappa, yy, tt) for tt in [0.5, 1.0, 1.5]])
    gtau = (1 + 2 * kappa * taus) ** 2
    slope = (1 + 4 * kappa * taus)
    left = [
        go.Scatter(x=geo[0], y=geo[1], mode="lines", line=dict(color="#1f77b4", width=1.5), name="法線方向の測地線"),
        go.Scatter(x=slices[0], y=slices[1], mode="lines", line=dict(color="#999", width=1), name="τ = 一定の断面"),
        go.Scatter(x=sigma[0], y=sigma[1], mode="lines", line=dict(color="#222", width=4), name="断面 Σ"),
    ]
    right = [
        go.Scatter(x=taus, y=gtau, mode="lines", line=dict(color="#d62728", width=3), name="g(τ)"),
        go.Scatter(x=taus, y=slope, mode="lines", line=dict(color="#444", dash="dot"), name="τ=0 での接線（傾き −2K）"),
    ]
    K = -2 * kappa
    title = f"κ = {kappa:+.2f}　K = −2κ = {K:+.2f}　∂τg(0) = −2K = {-2 * K:+.2f}"
    if kappa < 0:
        title += f"　焦点 τ = {1 / (2 * abs(kappa)):.2f}"
    return left + right, title


frames, steps = [], []
for kp in kappas:
    tr, title = build(kp)
    name = f"{kp:+.2f}"
    frames.append(go.Frame(data=tr, traces=[0, 1, 2, 3, 4], name=name, layout=go.Layout(title_text=title)))
    steps.append(dict(method="animate", label=name,
                      args=[[name], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))]))

k0 = int(np.argmin(np.abs(kappas - 0.4)))
tr0, title0 = build(kappas[k0])
fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5], horizontal_spacing=0.1,
                    subplot_titles=["(y, t) 時空図", "y = 0 の測地線の間隔 g(τ)"])
for t_ in tr0[:3]:
    fig.add_trace(t_, row=1, col=1)
for t_ in tr0[3:]:
    fig.add_trace(t_, row=1, col=2)
fig.frames = frames
fig.update_xaxes(title_text="y", range=[-1.2, 1.2], row=1, col=1)
fig.update_yaxes(title_text="t", range=[-0.6, 2.4], scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_xaxes(title_text="τ", range=[0, T], row=1, col=2)
fig.update_yaxes(title_text="g", range=[0, 7], row=1, col=2)
fig.update_layout(title_text=title0, sliders=[dict(active=k0, currentvalue=dict(prefix="κ = "), pad=dict(t=30), steps=steps)],
                  height=600, margin=dict(l=40, r=20, t=90, b=0), legend=dict(orientation="h", y=-0.28))
fig.show()

## 観察のポイント

- $\kappa>0$（断面が上に曲がる）では、測地線が外へ広がり、$g(\tau)$ は増えます。$\partial_\tau g>0$ なので、$K=-\tfrac12\partial_\tau g<0$ です。ノートのアニメ（anim06、$\kappa=0.4$）と同じ値 $K=-0.8$ です。
- $\kappa<0$ では、測地線が集まり、$g(\tau)$ が 0 に向かいます。
- $\kappa=0$（平らな断面）では、測地線は平行で、$g(\tau)=1$ のままです。$K=0$ です。
- 外的曲率 $K_{ij}$ は、「断面が、時間方向へどれだけ広がるか」を測る量です。付録Cの拘束条件と発展方程式は、この量 $K_{ij}$ を、時間発展する変数として扱います。